In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog qiskit-serverless
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# כתוב את תוכנית Qiskit Serverless הראשונה שלך

<details>
<summary><b>גרסאות חבילות</b></summary>

הקוד בדף זה פותח באמצעות הדרישות הבאות.
אנו ממליצים להשתמש בגרסאות אלו או בגרסאות חדשות יותר.

```
qiskit[all]~=1.3.1
qiskit-ibm-runtime~=0.34.0
qiskit-aer~=0.15.1
qiskit-serverless~=0.18.1
qiskit-ibm-catalog~=0.2
qiskit-addon-sqd~=0.8.1
qiskit-addon-utils~=0.1.0
qiskit-addon-mpf~=0.2.0
qiskit-addon-aqc-tensor~=0.1.2
qiskit-addon-obp~=0.1.0
scipy~=1.15.0
pyscf~=2.8.0
```
</details>
דוגמה זו מציגה כיצד להשתמש בכלי `qiskit-serverless` ליצירת תוכנית transpilation מקבילית, ולאחר מכן ליישם את `qiskit-ibm-catalog` כדי לפרוס את התוכנית ל-IBM Quantum Platform לשימוש כשירות מרוחק לשימוש חוזר.
## דוגמה: transpilation מרוחק עם Qiskit Serverless
התחל עם הדוגמה הבאה שמבצעת transpile ל-`circuit` בהתאם ל-`backend` נתון ורמת `optimization_level` יעד, והוסף בהדרגה עוד אלמנטים כדי לפרוס את עומס העבודה שלך ל-Qiskit Serverless.

הנח את תא הקוד הבא בקובץ `./source_files/transpile_remote.py`. קובץ זה הוא התוכנית שתועלה ל-Qiskit Serverless.

In [1]:
# This cell is hidden from users, it just creates a new folder
from pathlib import Path

Path("./source_files").mkdir(exist_ok=True)

In [2]:
%%writefile ./source_files/transpile_remote.py

from qiskit.transpiler import generate_preset_pass_manager

def transpile_remote(circuit, optimization_level, backend):
    """Transpiles an abstract circuit into an ISA circuit for a given backend."""
    pass_manager = generate_preset_pass_manager(
        optimization_level=optimization_level,
		backend=backend
    )
    isa_circuit = pass_manager.run(circuit)
    return isa_circuit

Writing ./source_files/transpile_remote.py


## Set up your files

Qiskit Serverless requires setting up your workload’s `.py` files into a dedicated directory. The following structure is an example of good practice:

```text
serverless_program
├── program_uploader.ipynb
└── source_files
    ├── transpile_remote.py
    └── *.py
```

Serverless uploads the contents of `source_files` to run remotely. Once these are set up, you can adjust `transpile_remote.py` to fetch inputs and return outputs.

### Get program arguments

Your initial `transpile_remote.py` has three inputs: `circuits`, `backend_name`, and `optimization_level`. Serverless is currently limited to only accept serializable inputs and outputs. For this reason, you cannot pass in `backend` directly, so use `backend_name` as a string instead.

In [3]:
%%writefile --append ./source_files/transpile_remote.py

from qiskit_serverless import get_arguments, save_result, distribute_task, get

# Get program arguments
arguments = get_arguments()
circuits = arguments.get("circuits")
backend_name = arguments.get("backend_name")
optimization_level = arguments.get("optimization_level")

Appending to ./source_files/transpile_remote.py


## הגדרת הקבצים שלך
Qiskit Serverless דורש הגדרת קבצי `.py` של עומס העבודה שלך לתוך תיקייה ייעודית. המבנה הבא הוא דוגמה לפרקטיקה טובה:

In [4]:
%%writefile --append ./source_files/transpile_remote.py

from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.backend(backend_name)

Appending to ./source_files/transpile_remote.py


Serverless מעלה את תוכן `source_files` להרצה מרחוק. לאחר הגדרתם, תוכל לכוונן את `transpile_remote.py` כדי לקבל קלטים ולהחזיר פלטים.

### קבלת ארגומנטים לתוכנית
ל-`transpile_remote.py` הראשוני שלך יש שלושה קלטים: `circuits`, `backend_name` ו-`optimization_level`. כרגע Serverless מוגבל לקבל רק קלטים ופלטים שניתן לסדרייה. מסיבה זו, אינך יכול להעביר את `backend` ישירות, לכן השתמש ב-`backend_name` כמחרוזת במקום.

In [5]:
%%writefile --append ./source_files/transpile_remote.py

results = [
    transpile_remote(circuit, 1, backend)
    for circuit in circuits
]

save_result({
    "transpiled_circuits": results
})

Appending to ./source_files/transpile_remote.py


## Deploy to IBM Quantum Platform

The previous section created a program to be run remotely. The code cells in this section upload that program to Qiskit Serverless.

Use `qiskit-ibm-catalog` to authenticate to `QiskitServerless` with your API key, which you can find on the [IBM Quantum dashboard](https://quantum.cloud.ibm.com), and upload the program.

You can use `save_account()` to save your credentials (See the [Set up to use IBM Cloud](/docs/guides/cloud-setup#cloud-save) section). Note that this writes your credentials to the same file as [`QiskitRuntimeService.save_account()`](/docs/api/qiskit-ibm-runtime/qiskit-runtime-service#save_account).

In [6]:
from qiskit_ibm_catalog import QiskitServerless, QiskitFunction

# Authenticate to the remote cluster and submit the pattern for remote execution
serverless = QiskitServerless()

בשלב זה, תוכל לקבל את ה-backend שלך עם `QiskitRuntimeService` ולהוסיף את התוכנית הקיימת שלך עם הקוד הבא. הקוד הבא דורש שכבר [שמרת את האישורים שלך](/guides/cloud-setup).

In [7]:
transpile_remote_demo = QiskitFunction(
    title="transpile_remote_serverless",
    entrypoint="transpile_remote.py",
    working_dir="./source_files/",
)

In [8]:
serverless.upload(transpile_remote_demo)

QiskitFunction(transpile_remote_serverless)

לבסוף, תוכל להריץ את `transpile_remote()` על כל ה-`circuits` שהועברו, ולהחזיר את `transpiled_circuits` כתוצאה:

In [9]:
# Get program from serverless.list() that matches the title of the one we uploaded
next(
    program
    for program in serverless.list()
    if program.title == "transpile_remote_serverless"
)

QiskitFunction(transpile_remote_serverless)

In [10]:
# This cell is hidden from users, it checks the program uploaded correctly
assert _.title == "transpile_remote_serverless"  # noqa: F821

In [11]:
# This cell is hidden from users, it checks the program executes correctly
from time import sleep
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
transpile_remote_serverless = serverless.load("transpile_remote_serverless")
job = transpile_remote_serverless.run(
    circuits=[qc],
    backend="ibm_sherbrooke",
    optimization_level=1,
)
while True:
    sleep(5)
    status = job.status()
    if status not in ["QUEUED", "INITIALIZING", "RUNNING", "DONE"]:
        raise Exception(
            f"Unexpected job status: '{status}'\n"
            + "Here are the logs:\n"
            + job.logs()
        )
    if status == "DONE":
        break

Qiskit Serverless דוחס את תוכן `working_dir` (במקרה זה, `source_files`) ל-`tar`, שמועלה ומנוקה לאחר מכן. ה-`entrypoint` מזהה את קובץ ההפעלה הראשי של התוכנית שעל Qiskit Serverless להריץ. בנוסף, אם לתוכנית שלך יש תלויות `pip` מותאמות אישית, תוכל להוסיף אותן למערך `dependencies`: